# 최적의 하이퍼파라미터로 Yolo testdataset 결과물 산출

In [ ]:
from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


In [ ]:
from pathlib import Path

EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
SUBMISSION_DIR = PROJECT_ROOT / "submissions"
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
DATA_DIR = PROJECT_ROOT / "data"
YOLO_DIR = DATA_DIR / "yolo_aug_clean"
CLASS_MAP = YOLO_DIR / "class_map.json"  # 전체 클래스(K-코드) 매핑

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"테스트 이미지: {KAGGLE_DATA / 'test_images'}")

In [ ]:
from src.inference import restrict_class_map, run_inference, save_submission
from src.utils import load_config
from src.data.kaggle_converter import convert_kaggle_to_coco

yolo_cfg = load_config(CONFIGS_DIR / "exp016_yolo11n_aug_clean_hpo_best.yaml")
EXP_NAME = yolo_cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / f"{EXP_NAME}_submission.csv"

kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}  # 제출 대상 56

# 56 클래스만 남긴 제출용 class_map
CLASS_MAP_SUBMIT = restrict_class_map(CLASS_MAP, allowed_ids)

preds = run_inference(
    weights_path=WEIGHTS,
    class_map_path=CLASS_MAP_SUBMIT,
    test_dir=KAGGLE_DATA / "test_images",
    conf_threshold=0.25,
    iou_threshold=0.45,
    img_size=yolo_cfg["data"]["img_size"],
)
save_submission(preds, SUBMISSION)
print("제출 파일:", SUBMISSION)


In [ ]:
import pandas as pd

sub_df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(sub_df["image_id"].unique())
bad = set(sub_df["category_id"].unique()) - allowed_ids
print(
    f"행 {len(sub_df)}, 예측 이미지 {len(pred_ids)}/{len(test_ids)}, "
    f"고유 category {sub_df['category_id'].nunique()}"
)
print("56 외 category_id?", bad if bad else "없음(정상)")
sub_df.head()
